# Cross-acquisition train/validation/test illustration

Seed42 development illustration by default. IMPORTANT: its default B_to_A recipe is numpy_original, but the later remaining-runs trainer uses sklearn recheck. Use saved memberships or set B_to_A to sklearn_recheck for those runs. Do not use an unverified reconstructed figure as proof of actual memberships.

Source: `gd_cross_acquisition_train_val_test_figure.py`.

Run in a fresh Kaggle GPU session with the appropriate dataset attached. Review the configuration near the top; original dataset/output paths are retained. Do not Run All across the repository.


In [ ]:
"""ONE CELL: DEVELOPMENT TRAIN + VALIDATION + UNSEEN TEST, BOTH DIRECTIONS.
Uses RGB filenames containing latitude/longitude. No split CSV, weights, training,
or 24 m buffered-split files are needed. Development membership depends on seed.
Source notebook: gd-second.ipynb, cross-acquisition cell (50 m GPS DBSCAN).
"""
from pathlib import Path
import json
import re
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.model_selection import train_test_split

RGB_DIR = None  # Automatic original dataset discovery; normally leave unchanged.
OUTPUT_DIR = None
DPI = 600
FIGSIZE = (17, 7.5)
GPS_PATTERN = re.compile(r"lat_(-?\d+(?:\.\d+)?)_lon_(-?\d+(?:\.\d+)?)", re.I)
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}


def read_gps_image_locations():
    preferred = [Path(p) for p in (
        "/kaggle/input/datasets/hemal7/gd-rgbms-db/RGB",
        "/kaggle/input/hemal7/gd-rgbms-db/RGB",
        "/kaggle/input/jarin19/rgbmsdb/RGB",
        "/kaggle/input/gd-rgbms-db/RGB", "/content/GD-RGBMS-DB/RGB")]
    if RGB_DIR is not None:
        preferred = [Path(RGB_DIR)]
    folders = [p for p in preferred if p.is_dir()]
    if not folders and RGB_DIR is None:
        for location in ("/kaggle/input", "/content"):
            parent = Path(location)
            if parent.exists():
                folders.extend(p for p in parent.rglob("RGB") if p.is_dir())
    candidates = []
    for folder in folders:
        rows = []
        for path in sorted(folder.rglob("*")):
            if not path.is_file() or path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue
            match = GPS_PATTERN.search(path.stem)
            if match:
                rows.append({"sample_name": path.stem, "latitude": float(match[1]),
                             "longitude": float(match[2])})
        if len(rows) == 400 and len({row['sample_name'] for row in rows}) == 400:
            candidates.append((pd.DataFrame(rows).sort_values('sample_name').reset_index(drop=True), folder))
    if not candidates:
        raise FileNotFoundError("The 400 GPS-labelled RGB images were not found. Attach GD-RGBMS-DB "
                                "or set RGB_DIR to its RGB directory. No CSV is required.")
    if len({tuple(frame.sample_name) for frame, folder in candidates}) != 1:
        raise ValueError("Different 400-image datasets were found. Set RGB_DIR to the actual study dataset.")
    frame, folder = candidates[0]
    if not (np.isfinite(frame[['latitude','longitude']].to_numpy()).all()
            and frame.latitude.between(-90,90).all() and frame.longitude.between(-180,180).all()):
        raise ValueError('Invalid GPS coordinates.')
    print('RGB directory:', folder)
    return frame, str(folder)


def assign_acquisition_groups(frame):
    frame = frame.copy()
    # Exact local-coordinate conversion and clustering settings in the
    # original cross-acquisition training cell of the second notebook.
    mean_lat, mean_lon = frame.latitude.mean(), frame.longitude.mean()
    frame['x_m'] = (frame.longitude - mean_lon) * 111320.0 * np.cos(np.deg2rad(mean_lat))
    frame['y_m'] = (frame.latitude - mean_lat) * 110540.0
    labels = DBSCAN(eps=50.0, min_samples=2).fit_predict(frame[['x_m','y_m']].to_numpy())
    sizes = {int(label): int((labels == label).sum()) for label in np.unique(labels)}
    if -1 in sizes or len(sizes) != 2 or sorted(sizes.values()) != [183,217]:
        raise ValueError(f'Expected two clean GPS groups of 217 and 183 images; found {sizes}.')
    group_a = next(label for label, count in sizes.items() if count == 217)
    frame['acquisition_group'] = np.where(labels == group_a, 'A', 'B')
    frame['final_role_when_training_on_A'] = np.where(frame.acquisition_group == 'A', 'train', 'unseen_test')
    frame['final_role_when_training_on_B'] = np.where(frame.acquisition_group == 'B', 'train', 'unseen_test')
    return frame



# Show the primary seed by default. To export all three: [42, 123, 2026].
PLOT_SEEDS = [42]
DEV_VAL_FRACTION = 0.20
# The supplied notebook contains two split implementations:
# A_to_B recheck cell: sklearn train_test_split; original bidirectional cell:
# NumPy default_rng permutation. Saved actual lists override these recipes.
DEV_METHODS = {'A_to_B': 'sklearn_recheck', 'B_to_A': 'numpy_original'}
# Optional exact run directories, if saved development image lists are available.
# Normally leave None; the known notebook output directories are checked below.
SPLIT_LIST_DIRS = {'A_to_B': None, 'B_to_A': None}


def canonical_name(value):
    name=str(value).strip().replace(chr(92),'/').split('/')[-1]
    return re.sub(r'\.(?:jpe?g|png|tiff?)$','',name,flags=re.I)


def validate_dev_lists(frame, source, train_names, val_names):
    source_names=set(frame.loc[frame.acquisition_group==source,'sample_name'])
    training=[canonical_name(name) for name in train_names]
    validation=[canonical_name(name) for name in val_names]
    expected_val=max(1,int(round(len(source_names)*DEV_VAL_FRACTION)))
    if (len(training)!=len(source_names)-expected_val or len(validation)!=expected_val
        or len(set(training))!=len(training) or len(set(validation))!=len(validation)
        or set(training)&set(validation) or set(training)|set(validation)!=source_names):
        raise ValueError(f'Development lists do not partition source Group {source} correctly.')
    return set(training),set(validation)


def development_membership(frame, source, seed):
    direction='A_to_B' if source=='A' else 'B_to_A'
    # The recheck cell leaves actual development names in the notebook namespace.
    if globals().get('SEED')==seed and all(key in globals() for key in ('dev_train_names','dev_val_names')):
        try:
            training,validation=validate_dev_lists(frame,source,globals()['dev_train_names'],globals()['dev_val_names'])
            return training,validation,'actual in-memory development image lists',False
        except (ValueError,TypeError):
            pass
    folders=[]
    explicit=SPLIT_LIST_DIRS.get(direction)
    if explicit is not None:
        folders=[Path(explicit)]
    else:
        for location in ('/kaggle/working','/content','/content/drive/MyDrive'):
            base=Path(location)
            if source=='A':
                folders.append(base/f'gd_cross_acquisition_A_to_B_seed{seed}_recheck')
            else:
                folders.append(base/f'gd_cross_acquisition_gatedfusion_seed{seed}'/'B_to_A')
    for folder in folders:
        paths=[folder/'development_train_list.txt',folder/'development_val_list.txt']
        if not all(path.is_file() for path in paths):
            if explicit is not None:
                raise FileNotFoundError(f'Actual development lists are missing from {folder}')
            continue
        config_path=folder/'experiment_configuration.json'
        if config_path.exists():
            config=json.loads(config_path.read_text())
            if config.get('seed') is not None and int(config['seed'])!=seed:
                if explicit is not None:
                    raise ValueError(f'Saved configuration seed differs from requested seed {seed}.')
                continue
        values=[[line.strip() for line in path.read_text(encoding='utf-8-sig').splitlines() if line.strip()]
                for path in paths]
        training,validation=validate_dev_lists(frame,source,*values)
        return training,validation,f'actual saved development lists: {folder}',False
    ordered=sorted(frame.loc[frame.acquisition_group==source,'sample_name'].tolist())
    n_val=max(1,int(round(len(ordered)*DEV_VAL_FRACTION)))
    method=DEV_METHODS[direction]
    if method=='sklearn_recheck':
        training,validation=train_test_split(ordered,test_size=n_val,random_state=seed,shuffle=True)
    elif method=='numpy_original':
        permutation=np.random.default_rng(seed).permutation(len(ordered))
        indices=set(permutation[:n_val].tolist())
        training=[name for index,name in enumerate(ordered) if index not in indices]
        validation=[name for index,name in enumerate(ordered) if index in indices]
    else:
        raise ValueError(f'Unknown development split method: {method}')
    training,validation=validate_dev_lists(frame,source,training,validation)
    return training,validation,f'reconstructed from shared notebook: {method}, seed {seed}',True


def make_development_figure(frame,seed):
    fig,axes=plt.subplots(1,2,figsize=FIGSIZE,sharex=True,sharey=True)
    xspan=max(float(frame.x_m.max()-frame.x_m.min()),1.)
    yspan=max(float(frame.y_m.max()-frame.y_m.min()),1.)
    memberships=frame.copy()
    records=[]
    for ax,source,target,panel in zip(axes,('A','B'),('B','A'),('a','b')):
        direction=f'{source}_to_{target}'
        train_names,val_names,origin,reconstructed=development_membership(frame,source,seed)
        roles=np.full(len(frame),'unseen_test',dtype=object)
        roles[frame.sample_name.isin(train_names).to_numpy()]='development_train'
        roles[frame.sample_name.isin(val_names).to_numpy()]='development_validation'
        memberships[f'{direction}_development_role']=roles
        styles=[('development_train','Development train','#0072B2','o',49),
                ('development_validation','Validation','#E69F00','^',88),
                ('unseen_test','Unseen test','#009E73','s',63)]
        for key,label,color,marker,size in styles:
            subset=frame[roles==key]
            ax.scatter(subset.x_m,subset.y_m,s=size,c=color,marker=marker,alpha=.90,
                       edgecolors='white',linewidths=.5,zorder=4 if key=='development_validation' else 3,
                       label=f'{label} (n = {len(subset)})')
        for group,subset in frame.groupby('acquisition_group'):
            ax.annotate(f'Group {group}',xy=(subset.x_m.median(),subset.y_m.max()),
                        xytext=(0,13),textcoords='offset points',ha='center',va='bottom',
                        fontsize=11,fontweight='bold',zorder=6,
                        bbox={'facecolor':'white','edgecolor':'#D9DEE5','alpha':.95,'pad':3.5})
        ax.set_xlim(frame.x_m.min()-.07*xspan,frame.x_m.max()+.07*xspan)
        ax.set_ylim(frame.y_m.min()-.12*yspan,frame.y_m.max()+.20*yspan)
        ax.set_xlabel('Local East–West position (m)',fontsize=12)
        ax.set_ylabel('Local North–South position (m)',fontsize=12)
        ax.set_title(f'({panel}) Develop on Group {source}; test on unseen Group {target}',
                     fontsize=12.5,fontweight='bold',pad=15)
        ax.tick_params(labelsize=10,labelleft=True)
        ax.ticklabel_format(style='plain',useOffset=False)
        ax.grid(True,alpha=.23,linewidth=.65)
        for spine in ax.spines.values():spine.set_color('#B8BDC5')
        ax.legend(loc='upper center',bbox_to_anchor=(.5,-.16),ncol=3,fontsize=9,
                  frameon=False,columnspacing=.8,handletextpad=.4)
        records.append({'direction':direction,'seed':seed,'source_group':source,'target_group':target,
                        'development_train':len(train_names),'validation':len(val_names),
                        'unseen_test':int((roles=='unseen_test').sum()),'membership_source':origin,
                        'reconstructed':reconstructed})
    fig.suptitle(f'Cross-acquisition development splits — seed {seed}',fontsize=15,fontweight='bold',y=.99)
    fig.text(.5,.018,'Validation uses source-group images only. After model selection, all source images are used for final refitting.',
             ha='center',fontsize=10,color='#50565F')
    fig.subplots_adjust(left=.065,right=.98,top=.86,bottom=.23,wspace=.23)
    return fig,memberships,records


def main():
    frame,rgb_source=read_gps_image_locations()
    frame=assign_acquisition_groups(frame)
    work=Path('/kaggle/working') if Path('/kaggle/working').exists() else (
        Path('/content') if Path('/content').exists() else Path.cwd())
    out=Path(OUTPUT_DIR) if OUTPUT_DIR is not None else work/'gd_cross_acquisition_validation_figures'
    out.mkdir(parents=True,exist_ok=True)
    all_records=[]
    for seed in PLOT_SEEDS:
        with plt.rc_context({'font.family':'DejaVu Sans','pdf.fonttype':42,'ps.fonttype':42,'svg.fonttype':'none'}):
            fig,memberships,records=make_development_figure(frame,int(seed))
            stem=f'cross_acquisition_train_val_test_seed{seed}'
            for extension in ('png','pdf','svg'):
                fig.savefig(out/f'{stem}.{extension}',dpi=DPI if extension=='png' else 300,
                            bbox_inches='tight',facecolor='white')
            plt.show()
            plt.close(fig)
        memberships.to_csv(out/f'image_memberships_seed{seed}.csv',index=False)
        all_records.extend(records)
        for record in records:
            print(f"{record['direction']} | seed {seed} | development train {record['development_train']} | "
                  f"validation {record['validation']} | unseen test {record['unseen_test']}")
            print('  Membership source:',record['membership_source'])
    pd.DataFrame(all_records).to_csv(out/'split_counts_and_sources.csv',index=False)
    (out/'figure_provenance.json').write_text(json.dumps({
        'rgb_source':rgb_source,'stage':'source-only development training/validation and held-out target test',
        'plot_seeds':PLOT_SEEDS,'fallback_split_methods':DEV_METHODS,'directional_records':all_records,
        'reconstruction_warning':'Without saved lists, these partitions follow the stated notebook recipes; '
                                 'their exact membership is not independently verified against the trained run.',
        'final_refit':'All source-group images, including former development-validation images, join final refitting.'
    },indent=2),encoding='utf-8')
    caption=('Cross-acquisition development partitions. (a) Group A supplies development training and validation '
             'images, while Group B is the unseen target test group. (b) The roles of Groups A and B are reversed. '
             'Blue circles, orange triangles and green squares denote development training, validation and '
             'unseen test images, respectively. Following source-only model and threshold selection, the model '
             'is reinitialized and refitted on all source-group images. The displayed validation partition is seed-specific.')
    if any(record['reconstructed'] for record in all_records):
        caption+=' Where saved lists were unavailable, memberships were reproduced using the specified notebook split recipes.'
    (out/'figure_caption.txt').write_text(caption,encoding='utf-8')
    archive=out/'cross_acquisition_train_val_test_figures.zip'
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as bundle:
        for path in sorted(out.iterdir()):
            if path.is_file() and path.suffix in {'.png','.pdf','.svg','.csv','.txt','.json'}:
                bundle.write(path,path.name)
    print('\nFigure folder:',out)
    print('Download ZIP:',archive)
    return out


if __name__=='__main__':
    main()
